# Phase 9: Prediction Validation & Error Analysis

Goals:
- compare predicted vs actual DA performance
- compute prediction accuracy
- model error analysis
- derive directions for improving the prediction model

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Path settings
ROOT = r'..'
BENCHMARK_DIR = os.path.join(ROOT, 'benchmark')
RESULTS_DIR = os.path.join(BENCHMARK_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print("="*100)
print("PHASE 9: Prediction Validation & Error Analysis")
print("="*100)
print(f"Results: {RESULTS_DIR}\n")

PHASE 9: Prediction Validation & Error Analysis
Results: ..\benchmark\results



In [2]:
# Load phase 7 and phase 8 results
print("Loading Phase 7 (Predictions) and Phase 8 (Actual Results)...\n")

try:
    with open(os.path.join(RESULTS_DIR, 'phase_07_transferability_profile.json'), 'r') as f:
        phase_07 = json.load(f)
    print("✓ Phase 7 loaded (Transferability Predictions)")
except FileNotFoundError:
    print("✗ Phase 7 not found")
    phase_07 = {}

try:
    with open(os.path.join(RESULTS_DIR, 'phase_08_da_benchmark.json'), 'r') as f:
        phase_08 = json.load(f)
    print("✓ Phase 8 loaded (Actual DA Results)")
except FileNotFoundError:
    print("✗ Phase 8 not found")
    phase_08 = {}

print()

Loading Phase 7 (Predictions) and Phase 8 (Actual Results)...

✓ Phase 7 loaded (Transferability Predictions)
✓ Phase 8 loaded (Actual DA Results)



In [3]:
# Prediction Validation Analysis
validation_results = {}

for prop in list(phase_07.keys()):
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    predicted = phase_07[prop]
    
    if prop not in phase_08:
        print(f"  No actual DA results for {prop}")
        continue
    
    actual = phase_08[prop]
    
    validation_results[prop] = {}
    
    # Extract key metrics
    pred_score = predicted.get('overall_score', None)
    pred_prob = predicted.get('da_success_probability', None)
    pred_rec = predicted.get('recommendation', None)
    
    actual_improve = actual.get('actual_improvement_pct', None)
    actual_success = actual.get('actual_success', None)
    
    print(f"\n  PREDICTIONS (Phase 7):")
    print(f"    Overall Score: {pred_score}/100")
    print(f"    DA Success Probability: {pred_prob:.0%}")
    print(f"    Recommendation: {pred_rec}")
    
    print(f"\n  ACTUAL RESULTS (Phase 8):")
    print(f"    R² Improvement: {actual_improve:.1f}%")
    print(f"    Actual Success (>5% improvement): {actual_success}")
    
    # Error Analysis
    print(f"\n  ERROR ANALYSIS:")
    
    # 1. Score Error
    # Convert actual improvement to estimated score (linear mapping)
    # 0% improvement = 40/100, 20% improvement = 80/100
    if actual_improve is not None:
        estimated_actual_score = min(100, max(0, 40 + actual_improve * 2))
        score_error = abs(pred_score - estimated_actual_score)
        score_error_pct = 100 * score_error / 100
        
        print(f"    Score MAE: {score_error:.1f} points ({score_error_pct:.1f}% of scale)")
        print(f"      Predicted score: {pred_score}/100")
        print(f"      Estimated actual score: {estimated_actual_score:.1f}/100")
        
        validation_results[prop]['score_mae'] = float(score_error)
        validation_results[prop]['score_mae_pct'] = float(score_error_pct)
    
    # 2. Probability Error
    if pred_prob is not None and actual_success is not None:
        actual_prob = 1.0 if actual_success else 0.0
        prob_error = abs(pred_prob - actual_prob)
        
        print(f"    Probability Error: {prob_error:.3f} ({prob_error*100:.1f}%)")
        print(f"      Predicted: {pred_prob:.0%}")
        print(f"      Actual: {actual_prob:.0%}")
        
        validation_results[prop]['prob_error'] = float(prob_error)
        validation_results[prop]['prob_error_pct'] = float(prob_error * 100)
    
    # 3. Classification Accuracy
    # Success threshold: prob > 50%
    if pred_prob is not None and actual_success is not None:
        pred_success = pred_prob > 0.5
        correct = (pred_success == actual_success)
        
        print(f"    Classification Accuracy: {'✓ Correct' if correct else '✗ Incorrect'}")
        print(f"      Predicted success: {pred_success}")
        print(f"      Actual success: {actual_success}")
        
        validation_results[prop]['classification_correct'] = bool(correct)
    
    # 4. Recommendation vs Reality
    print(f"\n  RECOMMENDATION VS REALITY:")
    print(f"    Predicted recommendation: {pred_rec}")
    
    # Map recommendation to expected improvement
    rec_to_expected = {
        'STRONGLY RECOMMENDED': (15, 25),
        'RECOMMENDED': (10, 20),
        'WORTH TRYING': (5, 15),
        'MARGINAL': (0, 5),
        'NOT RECOMMENDED': (-5, 0),
    }
    
    if pred_rec in rec_to_expected:
        expected_min, expected_max = rec_to_expected[pred_rec]
        expected_mid = (expected_min + expected_max) / 2
        
        in_range = expected_min <= actual_improve <= expected_max
        recommendation_accuracy = "✓ Within expected range" if in_range else "✗ Outside expected range"
        
        print(f"    Expected improvement: {expected_min}% - {expected_max}%")
        print(f"    Actual improvement: {actual_improve:.1f}%")
        print(f"    Status: {recommendation_accuracy}")
        
        validation_results[prop]['recommendation_accurate'] = bool(in_range)
        validation_results[prop]['expected_improvement_range'] = [expected_min, expected_max]
        validation_results[prop]['expected_vs_actual_error'] = float(abs(expected_mid - actual_improve))
    
    validation_results[prop]['pred_score'] = int(pred_score) if pred_score else None
    validation_results[prop]['pred_prob'] = float(pred_prob) if pred_prob else None
    validation_results[prop]['actual_improvement'] = float(actual_improve) if actual_improve else None
    validation_results[prop]['actual_success'] = bool(actual_success) if actual_success is not None else None


Property: CLEARANCE

  PREDICTIONS (Phase 7):
    Overall Score: 44/100
    DA Success Probability: 20%
    Recommendation: NOT RECOMMENDED

  ACTUAL RESULTS (Phase 8):
    R² Improvement: 3.6%
    Actual Success (>5% improvement): False

  ERROR ANALYSIS:
    Score MAE: 3.2 points (3.2% of scale)
      Predicted score: 44/100
      Estimated actual score: 47.2/100
    Probability Error: 0.200 (20.0%)
      Predicted: 20%
      Actual: 0%
    Classification Accuracy: ✓ Correct
      Predicted success: False
      Actual success: False

  RECOMMENDATION VS REALITY:
    Predicted recommendation: NOT RECOMMENDED
    Expected improvement: -5% - 0%
    Actual improvement: 3.6%
    Status: ✗ Outside expected range

Property: HALF_LIFE

  PREDICTIONS (Phase 7):
    Overall Score: 71/100
    DA Success Probability: 70%
    Recommendation: RECOMMENDED

  ACTUAL RESULTS (Phase 8):
    R² Improvement: -103.3%
    Actual Success (>5% improvement): False

  ERROR ANALYSIS:
    Score MAE: 71.0 poin

In [4]:
# Summary Statistics
print(f"\n\n{'='*100}")
print("PREDICTION VALIDATION SUMMARY")
print(f"{'='*100}\n")

# Calculate average errors
score_errors = [v['score_mae'] for v in validation_results.values() if 'score_mae' in v]
prob_errors = [v['prob_error'] for v in validation_results.values() if 'prob_error' in v]
classification_correct = [v['classification_correct'] for v in validation_results.values() if 'classification_correct' in v]
rec_accurate = [v['recommendation_accurate'] for v in validation_results.values() if 'recommendation_accurate' in v]

print("\nAVERAGE ERRORS:")
print(f"  Score MAE: {np.mean(score_errors):.1f} points" if score_errors else "  Score MAE: N/A")
print(f"  Probability Error: {np.mean(prob_errors)*100:.1f}%" if prob_errors else "  Probability Error: N/A")
print()

print("ACCURACY METRICS:")
print(f"  Classification Accuracy (Success/Fail): {np.mean(classification_correct)*100:.0f}%" if classification_correct else "  Classification Accuracy: N/A")
print(f"  Recommendation Accuracy (Within range): {np.mean(rec_accurate)*100:.0f}%" if rec_accurate else "  Recommendation Accuracy: N/A")
print()

print("\nPER-PROPERTY COMPARISON:")
summary_data = []
for prop, res in validation_results.items():
    row = {
        'Property': prop,
        'Pred_Score': f"{res.get('pred_score', 'N/A')}/100",
        'Actual_Improve': f"{res.get('actual_improvement', 'N/A'):.1f}%",
        'Score_Error': f"{res.get('score_mae', 'N/A'):.1f}",
        'Class_Correct': '✓' if res.get('classification_correct') else '✗' if 'classification_correct' in res else 'N/A',
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print()



PREDICTION VALIDATION SUMMARY


AVERAGE ERRORS:
  Score MAE: 28.8 points
  Probability Error: 56.7%

ACCURACY METRICS:
  Classification Accuracy (Success/Fail): 33%
  Recommendation Accuracy (Within range): 0%


PER-PROPERTY COMPARISON:
 Property Pred_Score Actual_Improve Score_Error Class_Correct
clearance     44/100           3.6%         3.2             ✓
half_life     71/100        -103.3%        71.0             ✗
       fu     48/100          10.1%        12.2             ✗



In [5]:
# Prediction Error Distribution Analysis
print(f"\n{'='*100}")
print("ERROR DISTRIBUTION ANALYSIS")
print(f"{'='*100}\n")

if score_errors:
    print(f"Score Error (MAE):")
    print(f"  Mean: {np.mean(score_errors):.1f} points")
    print(f"  Std: {np.std(score_errors):.1f} points")
    print(f"  Min: {np.min(score_errors):.1f} points")
    print(f"  Max: {np.max(score_errors):.1f} points")
    print(f"  Median: {np.median(score_errors):.1f} points")
    print()

if prob_errors:
    print(f"Probability Error (Absolute):")
    print(f"  Mean: {np.mean(prob_errors):.3f} ({np.mean(prob_errors)*100:.1f}%)")
    print(f"  Std: {np.std(prob_errors):.3f}")
    print(f"  Min: {np.min(prob_errors):.3f}")
    print(f"  Max: {np.max(prob_errors):.3f}")
    print(f"  Median: {np.median(prob_errors):.3f}")
    print()


ERROR DISTRIBUTION ANALYSIS

Score Error (MAE):
  Mean: 28.8 points
  Std: 30.1 points
  Min: 3.2 points
  Max: 71.0 points
  Median: 12.2 points

Probability Error (Absolute):
  Mean: 0.567 (56.7%)
  Std: 0.262
  Min: 0.200
  Max: 0.800
  Median: 0.700



In [6]:
# Model Validation Interpretation
print(f"\n{'='*100}")
print("MODEL VALIDATION INTERPRETATION")
print(f"{'='*100}\n")

mean_score_error = np.mean(score_errors) if score_errors else None
mean_prob_error = np.mean(prob_errors) if prob_errors else None
classification_acc = np.mean(classification_correct) if classification_correct else None

print("\nModel Performance Assessment:")
print()

if mean_score_error is not None:
    if mean_score_error < 10:
        print(f"  Score Prediction: EXCELLENT (MAE = {mean_score_error:.1f})")
        print(f"    → Transferability score is highly predictive")
    elif mean_score_error < 20:
        print(f"  Score Prediction: GOOD (MAE = {mean_score_error:.1f})")
        print(f"    → Transferability score predicts DA success reasonably well")
    elif mean_score_error < 30:
        print(f"  Score Prediction: MODERATE (MAE = {mean_score_error:.1f})")
        print(f"    → Some correlation but significant variance remains")
    else:
        print(f"  Score Prediction: POOR (MAE = {mean_score_error:.1f})")
        print(f"    → Score does not strongly predict actual results")
    print()

if classification_acc is not None:
    print(f"  Classification (Success/Failure): {classification_acc*100:.0f}% Accurate")
    if classification_acc >= 0.8:
        print(f"    → Excellent ability to predict success vs failure")
    elif classification_acc >= 0.6:
        print(f"    → Good ability to identify success cases")
    else:
        print(f"    → Limited predictive power for success/failure")
    print()

print("\nKey Findings:")
if mean_score_error and mean_score_error < 15:
    print("  ✓ Transferability framework shows strong predictive validity")
elif mean_score_error and mean_score_error < 25:
    print("  ✓ Framework provides useful guidance with moderate uncertainty")
else:
    print("  ✗ Prediction errors are large; model needs refinement")

if classification_acc and classification_acc >= 0.75:
    print("  ✓ Framework effectively separates viable from non-viable DA applications")
elif classification_acc and classification_acc >= 0.5:
    print("  ~ Framework has limited discriminative power")
else:
    print("  ✗ Framework cannot reliably predict success/failure")


MODEL VALIDATION INTERPRETATION


Model Performance Assessment:

  Score Prediction: MODERATE (MAE = 28.8)
    → Some correlation but significant variance remains

  Classification (Success/Failure): 33% Accurate
    → Limited predictive power for success/failure


Key Findings:
  ✗ Prediction errors are large; model needs refinement
  ✗ Framework cannot reliably predict success/failure


In [7]:
# Dimension-wise Error Analysis
print(f"\n{'='*100}")
print("DIMENSION-WISE CONTRIBUTION TO PREDICTION ERROR")
print(f"{'='*100}\n")

# Load phase 7 to analyze which dimensions contributed to errors
for prop in list(validation_results.keys()):
    if prop not in phase_07:
        continue
    
    profile = phase_07[prop]
    pred_score = profile.get('overall_score')
    actual = validation_results[prop].get('actual_improvement')
    
    if actual is not None and pred_score is not None:
        print(f"\n{prop.upper()}:")
        print(f"  Overall prediction error: {abs(pred_score - (40 + actual*2)):.1f} points")
        print(f"  Individual dimension scores:")
        
        scores = profile.get('individual_scores', {})
        if scores:
            for dim_name, score in sorted(scores.items(), key=lambda x: x[1]):
                status = "[LOW]" if score < 40 else "[MED]" if score < 70 else "[HIGH]"
                print(f"    {dim_name:25s}: {score:3d}/100 {status}")
            
            print(f"  \n  Bottleneck dimensions (score < 50):")
            bottlenecks = [(d, s) for d, s in scores.items() if s < 50]
            if bottlenecks:
                for dim, score in bottlenecks:
                    print(f"    → {dim}: {score}/100")
            else:
                print(f"    (None - all dimensions score >= 50)")


DIMENSION-WISE CONTRIBUTION TO PREDICTION ERROR


CLEARANCE:
  Overall prediction error: 3.2 points
  Individual dimension scores:
    Domain Discriminability  :  13/100 [LOW]
    Chemical Similarity      :  18/100 [LOW]
    Overlap Prevalence       :  30/100 [LOW]
    Label Correspondence     :  46/100 [MED]
    Dataset Compatibility    :  50/100 [MED]
    Representation Gap       :  95/100 [HIGH]
  
  Bottleneck dimensions (score < 50):
    → Chemical Similarity: 18/100
    → Label Correspondence: 46/100
    → Domain Discriminability: 13/100
    → Overlap Prevalence: 30/100

HALF_LIFE:
  Overall prediction error: 237.7 points
  Individual dimension scores:
    Chemical Similarity      :  18/100 [LOW]
    Domain Discriminability  :  43/100 [MED]
    Dataset Compatibility    :  80/100 [HIGH]
    Overlap Prevalence       :  90/100 [HIGH]
    Representation Gap       :  95/100 [HIGH]
    Label Correspondence     :  99/100 [HIGH]
  
  Bottleneck dimensions (score < 50):
    → Chemical Si

In [8]:
# Recommendations for Model Improvement
print(f"\n\n{'='*100}")
print("RECOMMENDATIONS FOR MODEL IMPROVEMENT")
print(f"{'='*100}\n")

print("""1. Data-driven weight optimization:
   Current weights (equal contribution model):
     - Chemical Similarity: 15%
     - Representation Gap: 20%
     - Label Correspondence: 20%
     - Domain Discriminability: 20%
     - Dataset Compatibility: 15%
     - Overlap Prevalence: 10%
   
   → Analyze which dimensions best predict actual improvement
   → Adjust weights to emphasize strongest predictors

2. Threshold calibration:
   → Recalibrate success probability thresholds
   → Use empirical data to tune score-to-probability mapping

3. Non-linear relationship modeling:
   → Test if score-improvement relationship is non-linear
   → Consider ensemble methods for prediction

4. Feature engineering:
   → Derive interaction terms between dimensions
   → Add property-specific adjustments

5. Uncertainty quantification:
   → Add confidence intervals to predictions
   → Provide prediction ranges instead of point estimates
""")



RECOMMENDATIONS FOR MODEL IMPROVEMENT

1. Data-driven weight optimization:
   Current weights (equal contribution model):
     - Chemical Similarity: 15%
     - Representation Gap: 20%
     - Label Correspondence: 20%
     - Domain Discriminability: 20%
     - Dataset Compatibility: 15%
     - Overlap Prevalence: 10%

   → Analyze which dimensions best predict actual improvement
   → Adjust weights to emphasize strongest predictors

2. Threshold calibration:
   → Recalibrate success probability thresholds
   → Use empirical data to tune score-to-probability mapping

3. Non-linear relationship modeling:
   → Test if score-improvement relationship is non-linear
   → Consider ensemble methods for prediction

4. Feature engineering:
   → Derive interaction terms between dimensions
   → Add property-specific adjustments

5. Uncertainty quantification:
   → Add confidence intervals to predictions
   → Provide prediction ranges instead of point estimates



In [9]:
# Save results
def convert_to_python_types(obj):
    if isinstance(obj, dict):
        return {k: convert_to_python_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_python_types(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj) if isinstance(obj, np.floating) else int(obj)
    return obj

# Create summary statistics
summary_stats = {
    'mean_score_error': float(np.mean(score_errors)) if score_errors else None,
    'std_score_error': float(np.std(score_errors)) if score_errors else None,
    'mean_prob_error': float(np.mean(prob_errors)) if prob_errors else None,
    'std_prob_error': float(np.std(prob_errors)) if prob_errors else None,
    'classification_accuracy': float(np.mean(classification_correct)) if classification_correct else None,
    'recommendation_accuracy': float(np.mean(rec_accurate)) if rec_accurate else None,
    'n_properties': len(validation_results),
}

with open(os.path.join(RESULTS_DIR, 'phase_09_prediction_validation.json'), 'w') as f:
    json.dump({
        'summary': convert_to_python_types(summary_stats),
        'by_property': convert_to_python_types(validation_results),
    }, f, indent=2)

print("\n\n✓ Results saved to benchmark/results/")
print(f"  - phase_09_prediction_validation.json")
print(f"\nPhase 9 Complete! DATAF Framework Validation Finished.")



✓ Results saved to benchmark/results/
  - phase_09_prediction_validation.json

Phase 9 Complete! DATAF Framework Validation Finished.
